# Trabajo Práctico — Predicción del uso de arma en robos en CABA
### Diplomatura en Ciencia de Datos y Análisis Avanzado — Entrega Final

**Pregunta de negocio.** Con recursos de patrullaje limitados, ¿en qué combinaciones *comuna × franja horaria*
conviene concentrar la presencia preventiva para cubrir la mayor proporción posible de **robos con arma**?

**Pregunta técnica.** ¿Aporta una red neuronal frente a modelos clásicos y, sobre todo, frente a
*baselines ingenuos* basados en promedios históricos?


## 1. Configuración y reproducibilidad

In [ ]:
import os, random, time, json, warnings, io
from pathlib import Path
from urllib.request import urlopen
import zipfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             precision_score, recall_score, f1_score, brier_score_loss,
                             roc_curve, mean_poisson_deviance)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping, log_evaluation
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

# FAST=1 solo sirve para probar el notebook rápido (menos épocas/réplicas). Resultados finales: FAST=0.
FAST = os.environ.get("FAST", "0") == "1"
N_BOOT = 40 if FAST else 300

# Fuente única: el commit exacto del repositorio que compartiste.
# Descargamos el ZIP completo del commit (en vez de raw.githubusercontent.com),
# porque la URL raw estaba devolviendo contenido vacío en Colab.
GITHUB_REPO = "https://codeload.github.com/charlyfriend68/DiploData/zip/3ba12474cac47d2e13bb776c1b15c21a81791d85"
REPO_COMMIT = "3ba12474cac47d2e13bb776c1b15c21a81791d85"
DATA_FOLDER = "DATOS"

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

# Descarga y abre el ZIP una sola vez; los CSV se leen directamente desde el ZIP.
print("Repositorio:", "https://github.com/charlyfriend68/DiploData/tree/" + REPO_COMMIT + "/" + DATA_FOLDER + "", "| FAST:", FAST, "| TF", tf.__version__)
try:
    with urlopen(GITHUB_REPO, timeout=120) as response:
        REPO_ZIP_BYTES = response.read()
    if not REPO_ZIP_BYTES:
        raise RuntimeError("GitHub devolvió un ZIP vacío.")
    REPO_ZIP = zipfile.ZipFile(io.BytesIO(REPO_ZIP_BYTES))
    REPO_FILES = REPO_ZIP.namelist()
    print("ZIP del commit descargado:", len(REPO_ZIP_BYTES), "bytes")
except Exception as exc:
    raise RuntimeError(
        "No se pudo descargar el ZIP del commit de GitHub. "
        "Comprobá que Colab tenga acceso a github.com/codeload.github.com. "
        f"Detalle: {type(exc).__name__}: {exc}"
    ) from exc

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

sns.set_style("whitegrid"); plt.rcParams["figure.dpi"] = 110
def savefig(name): plt.savefig(OUT / name, dpi=150, bbox_inches="tight")

print(
    "Repositorio:", REPO_COMMIT,
    "| ZIP:", len(REPO_ZIP_BYTES), "bytes",
    "| Archivos:", len(REPO_FILES),
    "| FAST:", FAST,
    "| TF:", tf.__version__
)

In [ ]:
# Diagnóstico: comprobar si el ZIP contiene los CSV reales
for nombre in REPO_FILES:
    if "delitos_" in nombre.lower():
        info = REPO_ZIP.getinfo(nombre)
        contenido = REPO_ZIP.read(nombre)

        print(
            nombre,
            "| tamaño:", info.file_size,
            "| primeros bytes:", repr(contenido[:100])
        )

## 2. Carga de datos (con control de calidad del parseo)

Cada archivo se lee con su separador detectado desde la cabecera y las fechas se parsean **por archivo**
con formato explícito (`%Y-%m-%d` o, si no, `%d/%m/%Y`). Si alguna fecha falla, el notebook se detiene.

In [ ]:
def load_year(year: int) -> pd.DataFrame:
    # Busca el archivo dentro del ZIP del commit exacto que compartiste.
    expected = f"DATOS/delitos_{year}.csv"
    matches = [name for name in REPO_FILES if name.endswith("/" + expected) or name == expected]
    if not matches:
        available = [name for name in REPO_FILES if "/" + DATA_FOLDER + "/" in name and "delitos_" in name]
        raise FileNotFoundError(
            f"No se encontró {expected} en el commit {REPO_COMMIT}. "
            f"Archivos de delitos encontrados en el ZIP: {available[:30]}"
        )

    file_name = matches[0]
    content = REPO_ZIP.read(file_name)
    if not content.strip():
        raise RuntimeError(f"El archivo {file_name} dentro del ZIP está vacío.")

    text = content.decode("utf-8-sig", errors="replace")
    if text.lstrip().startswith("version https://git-lfs.github.com/spec/v1"):
        raise RuntimeError(f"{file_name} es un puntero Git LFS, no contiene el CSV real.")

    header = next((line for line in text.splitlines() if line.strip()), "")
    sep = ";" if header.count(";") > header.count(",") else ","
    d = pd.read_csv(io.BytesIO(content), sep=sep, low_memory=False)
    d.columns = [str(c).strip().replace('"', "") for c in d.columns]
    required = ["fecha", "franja", "tipo", "subtipo", "uso_arma", "uso_moto", "comuna", "cantidad"]
    missing = [c for c in required if c not in d.columns]
    if missing:
        raise ValueError(
            f"{file_name} no contiene las columnas esperadas: {missing}. "
            f"Columnas encontradas: {d.columns.tolist()}"
        )

    f = pd.to_datetime(d["fecha"], format="%Y-%m-%d", errors="coerce")
    miss = f.isna()
    if miss.any():
        f[miss] = pd.to_datetime(d.loc[miss, "fecha"], format="%d/%m/%Y", errors="coerce")
    n_bad = int(f.isna().sum())
    assert n_bad == 0, f"{year}: {n_bad} fechas sin parsear"
    assert (f.dt.year == year).mean() > 0.999, f"{year}: fechas fuera del año del archivo"
    d["fecha"] = f

    keep = ["fecha", "franja", "tipo", "subtipo", "uso_arma", "uso_moto", "comuna", "cantidad"]
    d = d[keep].copy()
    d["archivo"] = year
    print(f"{year}: {len(d):>7,} filas | sep='{sep}' | {d.fecha.min().date()} -> {d.fecha.max().date()} | commit {REPO_COMMIT[:7]}")
    return d

df = pd.concat([load_year(y) for y in range(2016, 2026)], ignore_index=True)
print("\nTotal:", f"{len(df):,}", "filas")
# latitud/longitud se descartan a propósito: no se usan (cardinalidad alta) y el parseo de la v4 las corrompía.


## 3. EDA resumido y calidad de datos

In [ ]:
robos_all = df[df.tipo == "Robo"].copy()
print(f"Registros totales: {len(df):,} | Robos: {len(robos_all):,} ({len(robos_all)/len(df):.1%})")
print("cantidad == 1 en todos los robos:", bool((robos_all.cantidad == 1).all()))

nulos = robos_all[["comuna", "franja", "uso_arma"]].isna().mean().mul(100).round(2)
print("\n% de nulos en robos:\n", nulos.to_string())

robos = robos_all.dropna(subset=["comuna", "franja"]).copy()
robos["comuna"] = robos["comuna"].astype(int)
robos["franja"] = robos["franja"].astype(int)
robos["target"] = (robos["uso_arma"] == "SI").astype(int)
print(f"\nRobos utilizables (comuna y franja informadas): {len(robos):,} "
      f"(se descartan {len(robos_all)-len(robos):,} = {(1-len(robos)/len(robos_all)):.1%})")
print(f"Prevalencia global de uso de arma: {robos.target.mean():.2%}")
assert robos.fecha.max() >= pd.Timestamp("2025-06-01"), "2025 no se cargó completo"

# Los descartes por nulos ¿están sesgados hacia armados o no?
tmp = robos_all.assign(arma=(robos_all.uso_arma == "SI"), falta=robos_all.comuna.isna() | robos_all.franja.isna())
print("\nTasa de arma | con datos completos: %.2f%% | con comuna/franja faltante: %.2f%%" %
      (100*tmp[~tmp.falta].arma.mean(), 100*tmp[tmp.falta].arma.mean()))

anual = robos.groupby(robos.fecha.dt.year).agg(robos=("target", "size"), armados=("target", "sum"))
anual["prevalencia_%"] = (100 * anual.armados / anual.robos).round(2)
display(anual)

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
anual["robos"].plot(kind="bar", ax=ax[0], color="#4c72b0"); ax[0].set_title("Robos por año (2020 = pandemia; 2025: -30 % vs 2024)")
anual["prevalencia_%"].plot(kind="line", marker="o", ax=ax[1], color="#c44e52"); ax[1].set_title("% de robos con arma por año (deriva)")
by_fr = robos.groupby("franja").target.mean().mul(100)
by_fr.plot(kind="bar", ax=ax[2], color="#55a868"); ax[2].set_title("% con arma por franja horaria")
plt.tight_layout(); savefig("eda_anual_franja.png"); plt.show()

por_comuna = robos.groupby("comuna").agg(robos=("target", "size"), pct_arma=("target", "mean"))
por_comuna["pct_arma"] = (100 * por_comuna.pct_arma).round(1)
display(por_comuna.sort_values("pct_arma", ascending=False).T)

## 4. Ingeniería de variables **causales** y *tests* de no-fuga

**Se usa solo información disponible antes del hecho:** comuna, día de la semana, mes, hora (franja) y
antecedentes históricos de la misma combinación comuna × franja.

Los antecedentes se calculan sobre un **panel diario completo** (comuna × franja × día) con *lag* de un día:
para el día *t* solo se usan días ≤ *t−1*.

* `rate_12m`: tasa de uso de arma en los **últimos 365 días**, con *shrinkage* bayesiano hacia la tasa global
  de ese mismo período (evita tasas extremas en celdas con pocos casos).
* `rate_all`: lo mismo con ventana expansiva desde 2016 (lo que hacía la v4, ahora con lag y shrinkage).
* `log_vol_12m`: log(1 + robos de la combinación en 12 meses) → captura el **volumen**, clave para patrullar.
* El primer año (2016) es *warm-up* de las ventanas y **no se usa para entrenar**.

In [ ]:
M_SHRINK = 30           # "casos virtuales" del prior bayesiano
WINDOW = 365
MODEL_START = pd.Timestamp("2017-01-01")

comunas = np.sort(robos.comuna.unique()); C = len(comunas); F = 24
dates = pd.date_range(robos.fecha.min(), robos.fecha.max(), freq="D"); T = len(dates)
d0 = dates[0]

ci = (np.searchsorted(comunas, robos.comuna.values) * F + robos.franja.values)       # índice de combinación
di = (robos.fecha - d0).dt.days.values                                                 # índice de día
Nmat = np.zeros((C * F, T)); Kmat = np.zeros((C * F, T))
np.add.at(Nmat, (ci, di), 1)
np.add.at(Kmat, (ci, di), robos.target.values)

def lagged_window(M, w):
    """Suma de M en [t-w, t-1] para cada t (excluye el día t). Vectorizado por cumsum."""
    cum = M.cumsum(axis=1)
    upto_prev = cum - M                                   # acumulado hasta t-1
    shifted = np.zeros_like(cum); shifted[:, w+1:] = cum[:, :-(w+1)]   # acumulado hasta t-w-1
    return upto_prev - shifted

n12, k12 = lagged_window(Nmat, WINDOW), lagged_window(Kmat, WINDOW)
n_all, k_all = lagged_window(Nmat, T), lagged_window(Kmat, T)

# prior global (misma ventana, también con lag)
gN, gK = Nmat.sum(0, keepdims=True), Kmat.sum(0, keepdims=True)
g12n, g12k = lagged_window(gN, WINDOW)[0], lagged_window(gK, WINDOW)[0]
galln, gallk = lagged_window(gN, T)[0], lagged_window(gK, T)[0]
overall = Kmat.sum() / Nmat.sum()
p0_12 = np.where(g12n > 0, g12k / np.maximum(g12n, 1), overall)
p0_all = np.where(galln > 0, gallk / np.maximum(galln, 1), overall)

rate_12m = (k12 + M_SHRINK * p0_12) / (n12 + M_SHRINK)
rate_all = (k_all + M_SHRINK * p0_all) / (n_all + M_SHRINK)
logvol_12m = np.log1p(n12)
armday_12m = k12 / WINDOW                 # robos armados esperados por día según los últimos 12 meses
volday_12m = n12 / WINDOW

robos["rate_12m"] = rate_12m[ci, di]
robos["rate_all"] = rate_all[ci, di]
robos["log_vol_12m"] = logvol_12m[ci, di]

# ---- TEST DE CAUSALIDAD: recomputo a mano con datos estrictamente anteriores ----
rng = np.random.default_rng(SEED)
for j in rng.choice(len(robos), 25, replace=False):
    r = robos.iloc[j]
    prev = robos[(robos.comuna == r.comuna) & (robos.franja == r.franja) & (robos.fecha < r.fecha)]
    k_, n_ = prev.target.sum(), len(prev)
    gl = robos[robos.fecha < r.fecha]
    p0 = gl.target.mean() if len(gl) else overall
    manual = (k_ + M_SHRINK * p0) / (n_ + M_SHRINK)
    # p0_all del panel usa la tasa global acumulada hasta t-1: debe coincidir
    assert abs(manual - r.rate_all) < 1e-9, (j, manual, r.rate_all)
print("✔ Test de causalidad OK: rate_all coincide con un recómputo usando solo hechos anteriores al día del evento.")

## 5. Split temporal por **fecha** (70 / 15 / 15)

El corte se hace por fecha (nunca partiendo un mismo día entre conjuntos). Entrenamiento = pasado, validación
= intermedio (early stopping, calibración, umbral, selección), test = futuro (se mira una sola vez).

In [ ]:
ev = robos[robos.fecha >= MODEL_START].sort_values("fecha").reset_index(drop=True)
ev["mes"] = ev.fecha.dt.month.astype(str)
ev["dia"] = ev.fecha.dt.dayofweek.astype(str)
ev["comuna_c"] = ev.comuna.astype(str)
ev["franja_sin"] = np.sin(2*np.pi*ev.franja/24); ev["franja_cos"] = np.cos(2*np.pi*ev.franja/24)

d_tr = ev.fecha.iloc[int(len(ev)*0.70)]; d_va = ev.fecha.iloc[int(len(ev)*0.85)]
tr = ev[ev.fecha <= d_tr].copy(); va = ev[(ev.fecha > d_tr) & (ev.fecha <= d_va)].copy(); te = ev[ev.fecha > d_va].copy()
assert tr.fecha.max() < va.fecha.min() and va.fecha.max() < te.fecha.min()

for nm, s in [("Train", tr), ("Validation", va), ("Test", te)]:
    print(f"{nm:<11}{len(s):>8,}  [{s.fecha.min().date()} -> {s.fecha.max().date()}]  prevalencia={s.target.mean():.4f}")

CAT = ["mes", "dia", "comuna_c"]
NUM = ["franja", "franja_sin", "franja_cos", "rate_12m", "rate_all", "log_vol_12m"]
FEATS = CAT + NUM

def make_pre():
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
                              ("num", StandardScaler(), NUM)])

pre = make_pre().fit(tr[FEATS])
Xtr, Xva, Xte = (pre.transform(s[FEATS]) for s in (tr, va, te))
ytr, yva, yte = tr.target.values, va.target.values, te.target.values
print("\nDimensión tras preprocesar:", Xtr.shape)

n0, n1 = (ytr == 0).sum(), (ytr == 1).sum()
cw = {0: len(ytr)/(2*n0), 1: len(ytr)/(2*n1)}
print("class weights:", {k: round(v, 3) for k, v in cw.items()})

## 6. Baselines (lo primero que hay que superar)

Si un modelo de ML no supera claramente a **"usar el promedio histórico"**, no agrega valor.

In [ ]:
def rk(y, s): return roc_auc_score(y, s), average_precision_score(y, s)

baselines = {
    "B0 Prevalencia (azar)":              np.full(len(te), ytr.mean()),
    "B1 Tasa histórica expansiva":        te.rate_all.values,
    "B2 Tasa histórica 12 meses":         te.rate_12m.values,
}
rows = []
for k, s in baselines.items():
    a, p = rk(yte, s) if np.ptp(s) > 0 else (0.5, yte.mean())
    rows.append({"Baseline": k, "ROC-AUC": a, "PR-AUC": p})
base_df = pd.DataFrame(rows).round(4); display(base_df)
print(f"Prevalencia en test: {yte.mean():.4f}  → PR-AUC de un clasificador aleatorio.")

## 7. Modelos de clasificación P(arma | robo)

Regresión Logística, Random Forest, XGBoost, LightGBM y una **DNN** (una sola arquitectura, simple).
Hiperparámetros conservadores; la Regresión Logística y LightGBM tienen una grilla mínima **elegida en validación**.

In [ ]:
fit_time, models, p_va_raw, p_te_raw = {}, {}, {}, {}

# --- LR: grilla de C en validación (PR-AUC)
best = None
for Cc in [0.01, 0.1, 1.0]:
    m = LogisticRegression(C=Cc, max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
    sc = average_precision_score(yva, m.predict_proba(Xva)[:, 1])
    print(f"LR C={Cc:<5} PR-AUC val={sc:.4f}")
    if best is None or sc > best[0]: best = (sc, Cc)
t0 = time.time()
models["Regresión Logística"] = LogisticRegression(C=best[1], max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
fit_time["Regresión Logística"] = time.time() - t0
print("→ LR C elegido:", best[1])

# --- LightGBM: grilla de num_leaves en validación
best = None
for nl in [7, 15, 31]:
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=nl, min_child_samples=200, subsample=0.8,
                       subsample_freq=1, colsample_bytree=0.8, class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1)
    t0 = time.time(); m.fit(Xtr, ytr); dt = time.time() - t0
    sc = average_precision_score(yva, m.predict_proba(Xva)[:, 1])
    print(f"LGBM leaves={nl:<3} PR-AUC val={sc:.4f}")
    if best is None or sc > best[0]: best = (sc, nl, m, dt)
models["LightGBM"], fit_time["LightGBM"] = best[2], best[3]
LGBM_LEAVES = best[1]; print("→ LightGBM num_leaves elegido:", LGBM_LEAVES)

# --- Random Forest y XGBoost
t0 = time.time()
models["Random Forest"] = RandomForestClassifier(n_estimators=60 if FAST else 150, max_depth=12, min_samples_leaf=100, max_samples=0.5,
                                                 class_weight="balanced_subsample", n_jobs=-1, random_state=SEED).fit(Xtr, ytr)
fit_time["Random Forest"] = time.time() - t0
t0 = time.time()
models["XGBoost"] = XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                  min_child_weight=50, scale_pos_weight=n0/n1, eval_metric="logloss", tree_method="hist",
                                  n_jobs=-1, random_state=SEED).fit(Xtr, ytr)
fit_time["XGBoost"] = time.time() - t0

for k, m in models.items():
    p_va_raw[k] = m.predict_proba(Xva)[:, 1]; p_te_raw[k] = m.predict_proba(Xte)[:, 1]
    print(f"{k:<22} fit={fit_time[k]:7.1f}s  ROC val={roc_auc_score(yva, p_va_raw[k]):.4f}")

### DNN

Arquitectura `input → 32 → 8 → 1`. Con `class_weight` en el entrenamiento, **la validación también se pondera**
(`validation_data=(X, y, w)`), de modo que las curvas de pérdida de train y validación son comparables.

In [ ]:
def build_dnn(d):
    tf.keras.utils.set_random_seed(SEED)
    m = keras.Sequential([layers.Input(shape=(d,)), layers.Dense(32, activation="relu"),
                          layers.Dense(8, activation="relu"), layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc")])
    return m

w_tr = np.where(ytr == 1, cw[1], cw[0]).astype("float32")
w_va = np.where(yva == 1, cw[1], cw[0]).astype("float32")
dnn = build_dnn(Xtr.shape[1]); dnn.summary()
t0 = time.time()
hist = dnn.fit(Xtr, ytr, sample_weight=w_tr, validation_data=(Xva, yva, w_va),
               epochs=3 if FAST else 60, batch_size=512, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)])
fit_time["DNN"] = time.time() - t0
print(f"DNN: {len(hist.history['loss'])} épocas, {fit_time['DNN']:.0f}s")
p_va_raw["DNN"] = dnn.predict(Xva, verbose=0).ravel(); p_te_raw["DNN"] = dnn.predict(Xte, verbose=0).ravel()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist.history["loss"], label="Train (ponderada)"); ax[0].plot(hist.history["val_loss"], label="Validation (ponderada)")
ax[0].set_title("DNN — pérdida (ahora comparable)"); ax[0].set_xlabel("Época"); ax[0].legend()
ax[1].plot(hist.history["auc"], label="Train"); ax[1].plot(hist.history["val_auc"], label="Validation")
ax[1].set_title("DNN — AUC"); ax[1].set_xlabel("Época"); ax[1].legend()
plt.tight_layout(); savefig("dnn_curvas.png"); plt.show()

## 8. Calibración y umbral (ambos elegidos **sobre validación**)

`class_weight` infla las probabilidades, por lo que el umbral 0,5 no tiene significado. Calibramos con regresión
isotónica ajustada en validación y elegimos el umbral que maximiza F1 **en validación**; en test solo se aplica.

In [ ]:
NAMES = list(models.keys()) + ["DNN"]
iso, p_va, p_te, thr = {}, {}, {}, {}
for k in NAMES:
    iso[k] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_va_raw[k], yva)
    p_va[k], p_te[k] = iso[k].predict(p_va_raw[k]), iso[k].predict(p_te_raw[k])
    P, R, Th = precision_recall_curve(yva, p_va[k])
    f1 = 2*P[:-1]*R[:-1] / np.maximum(P[:-1] + R[:-1], 1e-12)
    thr[k] = Th[np.argmax(f1)]

rows = []
for k in NAMES:
    pred = (p_te[k] >= thr[k]).astype(int)
    rows.append({"Modelo": k,
                 "ROC-AUC": roc_auc_score(yte, p_te_raw[k]), "PR-AUC": average_precision_score(yte, p_te_raw[k]),
                 "Brier (cruda)": brier_score_loss(yte, p_te_raw[k]), "Brier (calibrada)": brier_score_loss(yte, p_te[k]),
                 "Umbral (val)": thr[k], "Precision": precision_score(yte, pred, zero_division=0),
                 "Recall": recall_score(yte, pred), "F1": f1_score(yte, pred), "Fit (s)": fit_time[k]})
res_df = pd.DataFrame(rows).set_index("Modelo")
display(res_df.round(4).sort_values("ROC-AUC", ascending=False))
print(f"Referencia: prevalencia en test = {yte.mean():.4f}; Brier de predecir siempre la prevalencia de train = "
      f"{brier_score_loss(yte, np.full(len(yte), ytr.mean())):.4f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for k in NAMES:
    fpr, tpr, _ = roc_curve(yte, p_te_raw[k]); ax[0].plot(fpr, tpr, label=f"{k} ({roc_auc_score(yte, p_te_raw[k]):.3f})")
    P, R, _ = precision_recall_curve(yte, p_te_raw[k]); ax[1].plot(R, P, label=k)
    pt, pp = [], []
    qs = np.quantile(p_te[k], np.linspace(0, 1, 11)); b = np.clip(np.digitize(p_te[k], qs[1:-1]), 0, 9)
    ax[2].plot([p_te[k][b == i].mean() for i in range(10)], [yte[b == i].mean() for i in range(10)], marker="o", ms=3, label=k)
ax[0].plot([0, 1], [0, 1], "--", c="gray"); ax[0].set_title("ROC (test)"); ax[0].legend(fontsize=8)
ax[1].axhline(yte.mean(), ls="--", c="gray"); ax[1].set_title("Precision-Recall (test)"); ax[1].set_xlabel("Recall")
lim = [0, max(0.4, yte.mean()*2)]; ax[2].plot(lim, lim, "--", c="gray"); ax[2].set_title("Calibración (post-isotónica, test)")
ax[2].set_xlabel("Probabilidad predicha"); ax[2].set_ylabel("Frecuencia observada")
plt.tight_layout(); savefig("roc_pr_calibracion.png"); plt.show()

## 9. ¿Son reales las diferencias? Bootstrap por bloques y *walk-forward*

**Bootstrap por bloques semanales** sobre el test (los robos de una misma semana están correlacionados, por
eso se remuestrean semanas completas, no filas). La misma réplica se aplica a todos los modelos → diferencias
**pareadas** contra la Regresión Logística. Luego, un **walk-forward** (5 pliegues expansivos) para ver la
estabilidad temporal.

In [ ]:
scores = {**{k: v for k, v in baselines.items() if "Prevalencia" not in k}, **{k: p_te_raw[k] for k in NAMES}}
week = ((te.fecha - te.fecha.min()).dt.days // 7).values
inv = np.unique(week, return_inverse=True)[1]; blocks = [np.where(inv == i)[0] for i in range(inv.max() + 1)]
rng = np.random.default_rng(SEED)
boot = {k: {"roc": [], "pr": []} for k in scores}
for _ in range(N_BOOT):
    pick = rng.integers(0, len(blocks), len(blocks)); idx = np.concatenate([blocks[i] for i in pick]); yy = yte[idx]
    if yy.min() == yy.max(): continue
    for k, s in scores.items():
        boot[k]["roc"].append(roc_auc_score(yy, s[idx])); boot[k]["pr"].append(average_precision_score(yy, s[idx]))

REF = "Regresión Logística"
rows = []
for k in scores:
    r, p = np.array(boot[k]["roc"]), np.array(boot[k]["pr"])
    dr, dp = r - np.array(boot[REF]["roc"]), p - np.array(boot[REF]["pr"])
    rows.append({"Modelo": k, "ROC-AUC": roc_auc_score(yte, scores[k]),
                 "IC95 ROC": f"[{np.percentile(r,2.5):.3f}, {np.percentile(r,97.5):.3f}]",
                 "PR-AUC": average_precision_score(yte, scores[k]), "IC95 PR": f"[{np.percentile(p,2.5):.3f}, {np.percentile(p,97.5):.3f}]",
                 "ΔROC vs LR": f"{dr.mean():+.4f} [{np.percentile(dr,2.5):+.4f}, {np.percentile(dr,97.5):+.4f}]",
                 "P(ΔROC>0)": (dr > 0).mean()})
boot_df = pd.DataFrame(rows).set_index("Modelo"); display(boot_df.round(4))
print(f"Réplicas: {len(boot[REF]['roc'])} sobre {len(blocks)} semanas de test. "
      "Si el IC de ΔROC incluye 0, la diferencia con la Regresión Logística NO es distinguible del ruido.")

# --- Walk-forward sobre el período pre-test (train+val) para LR y LightGBM
pre_ev = ev[ev.fecha <= d_va].reset_index(drop=True)
mk = {"Regresión Logística": lambda: LogisticRegression(C=models[REF].C, max_iter=2000, class_weight="balanced", random_state=SEED),
      "LightGBM": lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=LGBM_LEAVES, min_child_samples=200, subsample=0.8,
                                         subsample_freq=1, colsample_bytree=0.8, class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1)}
wf = []
for name, f in mk.items():
    for i, (a, b) in enumerate(TimeSeriesSplit(n_splits=3 if FAST else 5).split(pre_ev)):
        pp = Pipeline([("pre", make_pre()), ("m", f())]).fit(pre_ev.iloc[a][FEATS], pre_ev.iloc[a].target)
        s = pp.predict_proba(pre_ev.iloc[b][FEATS])[:, 1]; yb = pre_ev.iloc[b].target.values
        wf.append({"Modelo": name, "Pliegue": i + 1, "Hasta": pre_ev.iloc[b].fecha.max().date(), "ROC-AUC": roc_auc_score(yb, s),
                   "PR-AUC": average_precision_score(yb, s), "Prevalencia": yb.mean(),
                   "ROC baseline B2": roc_auc_score(yb, pre_ev.iloc[b].rate_12m)})
wf_df = pd.DataFrame(wf); display(wf_df.round(4))
display(wf_df.groupby("Modelo")[["ROC-AUC", "PR-AUC", "ROC baseline B2"]].agg(["mean", "std"]).round(4))

## 10. Valor de negocio: del "riesgo por robo" a la **cobertura de robos armados**

Un modelo de P(arma | robo) **no** dice dónde habrá más robos armados: falta el volumen. Se comparan estrategias
para elegir el **20 % de las 360 combinaciones comuna × franja** (72 celdas). Todas las estrategias usan solo
información **previa al inicio del test** (ranking fijo, como en un despliegue real) y se evalúan con los robos
armados **reales** del test:

| Estrategia | Score por celda |
|---|---|
| Azar | — (simulación) |
| Volumen histórico | robos totales de los últimos 12 meses |
| **Armados históricos** | robos armados de los últimos 12 meses *(baseline fuerte)* |
| Tasa histórica | `rate_12m` |
| Modelo P | media de P(arma\|robo) en la grilla de test *(lo que hacía la v4)* |
| **Modelo P × volumen** | Σ P(arma\|robo) × volumen esperado diario |
| **Poisson** | LightGBM Poisson sobre el panel diario (cuenta esperada de armados) |
| Oráculo | robos armados reales (techo teórico) |

In [ ]:
# ---------- Panel diario (comuna × franja × día) para scoring sin mirar el volumen futuro
T_ev0 = (MODEL_START - d0).days
def panel_slice(t_from, t_to):
    ts = np.arange(t_from, t_to + 1); nT = len(ts)
    p = pd.DataFrame({"combo": np.repeat(np.arange(C*F), nT), "t": np.tile(ts, C*F)})
    p["comuna_c"] = comunas[p.combo // F].astype(str); p["franja"] = p.combo % F
    dd = d0 + pd.to_timedelta(p.t, unit="D"); p["mes"] = dd.dt.month.astype(str); p["dia"] = dd.dt.dayofweek.astype(str)
    p["franja_sin"] = np.sin(2*np.pi*p.franja/24); p["franja_cos"] = np.cos(2*np.pi*p.franja/24)
    for nm, M in [("rate_12m", rate_12m), ("rate_all", rate_all), ("log_vol_12m", logvol_12m), ("armday_12m", armday_12m), ("volday_12m", volday_12m)]:
        p[nm] = M[p.combo.values, p.t.values]
    p["n"] = Nmat[p.combo.values, p.t.values]; p["k"] = Kmat[p.combo.values, p.t.values]
    return p

t_tr_end, t_va_end = (d_tr - d0).days, (d_va - d0).days
t_te0, t_te1 = t_va_end + 1, T - 1
grid_te = panel_slice(t_te0, t_te1)
print(f"Test: {len(grid_te):,} filas de panel ({t_te1 - t_te0 + 1} días × {C*F} celdas)")

# ---------- Poisson (cuenta esperada de robos armados por celda-día)
PF = ["comuna_c", "franja", "mes", "dia", "rate_12m", "rate_all", "log_vol_12m", "armday_12m"]
ptr, pva = panel_slice(T_ev0, t_tr_end), panel_slice(t_tr_end + 1, t_va_end)
CATS = {"comuna_c": [str(c) for c in comunas], "mes": [str(m) for m in range(1, 13)], "dia": [str(d) for d in range(7)]}
for d_ in (ptr, pva, grid_te):
    for c_, lv in CATS.items(): d_[c_] = pd.Categorical(d_[c_], categories=lv)
pois = LGBMRegressor(objective="poisson", n_estimators=1000 if not FAST else 100, learning_rate=0.03, num_leaves=31, min_child_samples=200,
                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1, n_jobs=-1)
pois.fit(ptr[PF], ptr.k, eval_set=[(pva[PF], pva.k)], callbacks=[early_stopping(50, verbose=False)])
grid_te["pois"] = np.clip(pois.predict(grid_te[PF]), 1e-9, None)
dev_model = mean_poisson_deviance(grid_te.k, grid_te.pois)
dev_base = mean_poisson_deviance(grid_te.k, np.clip(grid_te.armday_12m, 1e-4, None))
print(f"Poisson best_iter={pois.best_iteration_} | deviance test: modelo={dev_model:.4f} vs. baseline 'armados/día 12m'={dev_base:.4f}")

# ---------- Scores por celda
def per_combo(series): return series.groupby(grid_te.combo.values).sum().reindex(range(C*F)).values
for k in NAMES:
    Xg = pre.transform(grid_te[CAT + NUM].astype({"comuna_c": str, "mes": str, "dia": str}))
    raw = (dnn.predict(Xg, verbose=0).ravel() if k == "DNN" else models[k].predict_proba(Xg)[:, 1])
    grid_te["p_" + k] = iso[k].predict(raw)

last = t_te0
score_cells = {
    "Volumen histórico (12m)":    n12[:, last],
    "Armados históricos (12m)":   k12[:, last],
    "Tasa histórica (12m)":       rate_12m[:, last],
    "Poisson (cuenta esperada)":  per_combo(grid_te.pois),
}
for k in NAMES:
    score_cells[f"Modelo P — {k}"] = per_combo(grid_te["p_" + k]) / (t_te1 - t_te0 + 1)
    score_cells[f"Modelo P×volumen — {k}"] = per_combo(grid_te["p_" + k] * grid_te.volday_12m)
k_true = Kmat[:, t_te0:t_te1 + 1]; k_cell = k_true.sum(1); total_k = k_cell.sum()
score_cells["Oráculo (techo)"] = k_cell

n_top = int(np.ceil(0.20 * C * F))
def coverage(score): return k_cell[np.argsort(-score, kind="stable")[:n_top]].sum() / total_k
rand = np.array([k_cell[rng.permutation(C*F)[:n_top]].sum() / total_k for _ in range(2000)])

# IC por bloques semanales (ranking fijo, se remuestrean semanas)
wk = (np.arange(k_true.shape[1]) // 7); W = wk.max() + 1
Kw = np.stack([k_true[:, wk == w].sum(1) for w in range(W)], 1)
def cov_ci(score, B=2000):
    S = np.argsort(-score, kind="stable")[:n_top]; cw, tw = Kw[S].sum(0), Kw.sum(0)
    pk = rng.integers(0, W, (B, W)); v = cw[pk].sum(1) / tw[pk].sum(1)
    return np.percentile(v, [2.5, 97.5])

rows = [{"Estrategia": "Azar (media de 2000 sorteos)", "Cobertura top-20%": rand.mean(), "IC95": f"[{np.percentile(rand,2.5):.3f}, {np.percentile(rand,97.5):.3f}]"}]
for k, s in score_cells.items():
    lo, hi = cov_ci(s); rows.append({"Estrategia": k, "Cobertura top-20%": coverage(s), "IC95": f"[{lo:.3f}, {hi:.3f}]"})
cov_df = pd.DataFrame(rows).set_index("Estrategia"); display(cov_df.round(4))

# ---- Diferencias PAREADAS contra el baseline fuerte "Armados históricos (12m)"
def cov_diff(sA, sB, B=4000):
    A = np.argsort(-sA, kind="stable")[:n_top]; Bi = np.argsort(-sB, kind="stable")[:n_top]
    pk = rng.integers(0, W, (B, W)); tw = Kw.sum(0)
    d = (Kw[A].sum(0)[pk].sum(1) - Kw[Bi].sum(0)[pk].sum(1)) / tw[pk].sum(1)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), (d > 0).mean()
BASE_STR = "Armados históricos (12m)"
pair_rows = []
for k in ["Volumen histórico (12m)", "Tasa histórica (12m)", f"Modelo P — {REF}", f"Modelo P×volumen — {REF}", "Modelo P×volumen — LightGBM", "Poisson (cuenta esperada)"]:
    m_, lo_, hi_, pg_ = cov_diff(score_cells[k], score_cells[BASE_STR])
    pair_rows.append({"Estrategia": k, "Δ cobertura vs armados históricos (pp)": 100*m_, "IC95 (pp)": f"[{100*lo_:+.2f}, {100*hi_:+.2f}]", "P(Δ>0)": pg_})
pair_df = pd.DataFrame(pair_rows).set_index("Estrategia"); display(pair_df.round(3))

print(f"Robos armados en test: {int(total_k):,} | celdas priorizadas: {n_top} de {C*F}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
show = ["Volumen histórico (12m)", "Armados históricos (12m)", "Tasa histórica (12m)", "Poisson (cuenta esperada)",
        f"Modelo P — {REF}", f"Modelo P×volumen — {REF}", "Oráculo (techo)"]
for nm in show:
    o = np.argsort(-score_cells[nm], kind="stable"); cum = np.cumsum(k_cell[o]) / total_k
    ax.plot(np.arange(1, C*F+1) / (C*F) * 100, cum * 100, label=nm, lw=2.2 if "Poisson" in nm or "×" in nm else 1.4)
ax.plot([0, 100], [0, 100], "--", c="gray", label="Azar"); ax.axvline(20, c="red", ls=":")
ax.set_xlabel("% de celdas comuna × franja priorizadas"); ax.set_ylabel("% de robos con arma cubiertos (test)")
ax.set_title("Curva de ganancia acumulada — estrategias de priorización"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); savefig("ganancia_acumulada.png"); plt.show()

top = np.argsort(-score_cells["Poisson (cuenta esperada)"], kind="stable")[:15]
tbl = pd.DataFrame({"comuna": comunas[top // F], "franja": top % F, "armados esperados/día": score_cells["Poisson (cuenta esperada)"][top] / (t_te1 - t_te0 + 1),
                    "armados reales (test)": k_cell[top].astype(int)})
print("Top-15 celdas según el modelo Poisson:"); display(tbl.round(4).reset_index(drop=True))

## 11. Análisis de errores segmentado (modelo de referencia: Regresión Logística, umbral de validación)

Un modelo con buen desempeño global puede fallar sistemáticamente en ciertos territorios u horarios;
eso tiene consecuencias de equidad y de operación.

In [ ]:
seg = te[["comuna", "franja", "dia", "target"]].copy(); seg["pred"] = (p_te[REF] >= thr[REF]).astype(int); seg["p"] = p_te_raw[REF]
seg["bloque"] = pd.cut(seg.franja, [-1, 5, 11, 17, 23], labels=["Madrugada 0-5", "Mañana 6-11", "Tarde 12-17", "Noche 18-23"])
def tabla(g):
    r = []
    for key, s in seg.groupby(g):
        if s.target.sum() == 0: continue
        r.append({g: key, "N": len(s), "% armados": 100*s.target.mean(), "Recall": recall_score(s.target, s.pred),
                  "Precision": precision_score(s.target, s.pred, zero_division=0),
                  "ROC-AUC (intra-grupo)": roc_auc_score(s.target, s.p) if s.target.nunique() > 1 else np.nan})
    return pd.DataFrame(r).round(3)
t_com = tabla("comuna").sort_values("N", ascending=False); display(t_com)
display(tabla("bloque"))
fig, ax = plt.subplots(figsize=(10, 3.8)); t_c = t_com.sort_values("comuna")
ax.bar(t_c.comuna.astype(str), t_c.Recall, color="#4c72b0"); ax.set_title("Recall por comuna (disparidad de desempeño)"); ax.set_xlabel("Comuna")
plt.tight_layout(); savefig("recall_por_comuna.png"); plt.show()

## 12. Síntesis y exportación de resultados

In [ ]:
print("="*78); print("RESUMEN v5"); print("="*78)
print(f"Datos: {len(robos):,} robos utilizables 2016-{robos.fecha.max().date()} | modelado desde {MODEL_START.date()}")
print(f"Split: train hasta {d_tr.date()} | val hasta {d_va.date()} | test hasta {te.fecha.max().date()} (n={len(te):,}, prev={yte.mean():.3%})")
print("\nClasificación (test):"); print(res_df[["ROC-AUC", "PR-AUC", "Recall", "Precision", "F1"]].round(4).to_string())
print("\nBaselines:"); print(base_df.to_string(index=False))
print("\nCobertura top-20% celdas:"); print(cov_df.round(4).to_string())

results = {"config": {"seed": SEED, "fast": FAST, "model_start": str(MODEL_START.date()), "d_train_end": str(d_tr.date()), "d_val_end": str(d_va.date()),
                      "test_end": str(te.fecha.max().date()), "shrink": M_SHRINK, "window": WINDOW, "n_boot": N_BOOT},
           "datos": {"robos_utilizables": int(len(robos)), "n_test": int(len(te)), "prev_train": float(ytr.mean()), "prev_val": float(yva.mean()), "prev_test": float(yte.mean())},
           "clasificacion": res_df.round(5).reset_index().to_dict("records"), "baselines": base_df.to_dict("records"),
           "bootstrap": boot_df.reset_index().astype(str).to_dict("records"), "walk_forward": wf_df.astype(str).to_dict("records"),
           "cobertura": cov_df.round(5).reset_index().to_dict("records"), "poisson_deviance": {"modelo": float(dev_model), "baseline": float(dev_base)}}
(OUT / "resultados_v5.json").write_text(json.dumps(results, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
res_df.to_csv(OUT / "metricas_clasificacion.csv"); cov_df.to_csv(OUT / "cobertura_estrategias.csv"); t_com.to_csv(OUT / "errores_por_comuna.csv", index=False)
print("\n✔ Exportado en ./outputs (JSON, CSV y figuras PNG para el informe)")

## 12a. Conclusiones (generadas a partir de los resultados de esta corrida)

1. **Techo predictivo bajo.** Los cinco modelos logran ROC-AUC 0.690–0.692 y PR-AUC 0.241–0.243 (un clasificador al azar daría PR-AUC = 0.115). Las diferencias entre modelos están dentro del ruido (los IC95 pareados de ΔROC contra la Regresión Logística incluyen 0; Random Forest es levemente peor). **La DNN no aporta:** mismo desempeño con 29 s de entrenamiento contra 2.0 s de la Regresión Logística, y sin interpretabilidad.

2. **El baseline casi iguala al ML.** Usar directamente la tasa histórica de 12 meses por comuna × franja da ROC-AUC 0.6905, contra 0.6922 de la Regresión Logística. En el walk-forward son prácticamente idénticos (0.6596 vs 0.6595). Casi toda la señal está en el historial de la celda; el resto de las variables y la complejidad del modelo agregan muy poco.

3. **Valor de negocio (72 de 360 celdas priorizadas; cobertura de los 8,878 robos armados de test).** Azar 20.0 %
*   **Volumen histórico 39.6 %**
*   **Armados históricos 12m 51.7 %**
*   **Ranking por P(arma|robo) 45.9 %**
*   **P × volumen 53.1 %**
*   **Poisson 53.3 %**
*   **Techo teórico de un ranking fijo 53.5 %.**

    La cifra de la v4 (~44 %) era correcta, pero se comparaba solo contra el azar: contra un baseline razonable ("seguir los robos armados del último año"), rankear solo por P(arma|robo) queda a -5.8 pp. Al incorporar el volumen, el modelo queda a +1.4 pp del baseline fuerte (IC95 [+0.80, +1.95]), prácticamente en el techo del ranking fijo. El modelo Poisson no mejora la deviance del baseline simple (0.2681 vs 0.2651; menor es mejor).

4. **Equidad territorial.** Con un único umbral, el recall es 0 en 8 de 15 comunas; el modelo detecta casi todo en las comunas 8 y 4 y casi nada en las demás. Funciona, en la práctica, como un clasificador de comuna. Hay que fijar umbrales o metas por territorio antes de operar.

5. **Deriva.** La prevalencia de armas cayó de 19.3% (train) a 12.9% (validation) y 11.5% (test); la calibración isotónica en validación (Brier 0.176 → 0.095) es indispensable y debe repetirse periódicamente.

Recomendación

*   **No usar la DNN.** Para producción: **Regresión Logística calibrada × volumen esperado** (o, por simplicidad, el ranking de armados históricos de 12 meses), con reentrenamiento y recalibración mensual.
*   La mejora del ML sobre el baseline es chica: el aporte principal del proyecto es haber medido que el problema se resuelve casi por completo con estadística descriptiva bien construida.
*   Antes de escalar, medir el efecto real del patrullaje con un piloto con zonas de control.



## 13. Etica, privacidad y limitaciones (requisito de la consigna)

* **Subregistro / sesgo de medición.** Los datos son *denuncias*, no delitos. Zonas con menor propensión a denunciar
  aparecen como "seguras". El modelo aprende el comportamiento de denuncia y de registro, no la realidad criminal.
* **Bucle de retroalimentación.** Más patrullaje → más detecciones y registros en esa zona → más riesgo estimado.
  Mitigación: piloto con zonas de control, ranking con rotación/aleatorización parcial y reentrenamiento auditado.
* **Estigmatización territorial.** Las comunas con mayor proporción de robos armados concentran la atención. El
  informe comunica riesgo **por celda comuna × franja**, nunca por personas ni barrios con nombre propio.
* **Equidad de desempeño.** El recall varía fuertemente entre comunas (§11). Antes de operar hay que fijar un
  umbral mínimo de desempeño por territorio o usar umbrales diferenciales.
* **Privacidad.** Se usan datos públicos y agregados, sin datos personales. Se descartan latitud/longitud y barrio a
  propósito: nivel geográfico grueso (comuna) para evitar reidentificación y vigilancia hiper-localizada.
* **Deriva (*drift*).** La prevalencia de armas cambió de ~21 % (2017-2022) a ~12 % (2024-25). Se usa ventana de 12 meses,
  calibración y se recomienda monitoreo mensual de Brier/recalibración.
* **Datos incompletos.** ~0,9 % de robos sin comuna/franja (descartados; su tasa de arma es parecida) ; 2025 está completo por mes pero registra ~30 % menos robos que 2024 (¿caída real o cambio de registro?); 2020 es atípico (pandemia).
* **Alcance causal.** Todo es asociación predictiva. La efectividad del patrullaje debe medirse con un diseño
  causal (despliegue escalonado con controles), no inferirse del modelo.
* **Gobernanza.** Responsable humano de la decisión, bitácora de versiones del modelo, auditoría trimestral de
  sesgos y publicación de métricas por comuna.